# Student Depression Prediction — ML Zoomcamp midterm project

**Problem.** Predict whether a student reports depression (`Depression` = 1) from survey answers about
academic pressure, sleep, diet, finances and similar factors. A model like this could help a student
counselling service prioritise outreach or build a quick self-screening questionnaire. It is **not** a
diagnostic tool.

**Data.** [Student Depression Dataset](https://www.kaggle.com/datasets/hopesb/student-depression-dataset)
(Kaggle, 27,901 rows, 18 columns; the dataset is tagged as synthetic on Kaggle).

**Metric.** The classes are mildly imbalanced (about 58 / 42), so models are selected by **ROC AUC** (5-fold CV)
and we also report accuracy, precision, recall and F1 on a held-out test set. Recall matters most for a
screening use case (missing an at-risk student is worse than a false alarm).

This notebook covers the data cleaning, EDA and model selection. The final pipeline is produced by `train.py`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

sys.path.insert(0, str(Path.cwd()))
import features as F

sns.set_theme(style="whitegrid")
raw = pd.read_csv(F.DATA_PATH)
print(raw.shape)
raw.head()

## 1. Data cleaning

In [ ]:
print("Missing values:"); print(raw.isna().sum()[raw.isna().sum() > 0])
print("\nDuplicate rows:", raw.duplicated().sum())
print("\nProfession counts (top 5):"); print(raw["Profession"].value_counts().head(5))
print("\nNon-zero rows in near-constant columns:")
print(raw[["Work Pressure", "Job Satisfaction"]].apply(lambda s: (s != 0).sum()))

Cleaning decisions (implemented in `features.load_data`):

* keep only `Profession == "Student"` (31 rows are other professions) and drop the column,
* drop `id` (identifier) and `Work Pressure` / `Job Satisfaction` (zero for almost every student),
* rename columns to `snake_case` (these are also the JSON field names of the API),
* the 3 missing `financial_stress` values are imputed with the median **inside the pipeline**, so the
  API can handle a missing value the same way.

In [ ]:
df = F.load_data()
print(df.shape)
df.head()

## 2. Exploratory data analysis

In [ ]:
print(df[F.TARGET].value_counts(normalize=True).round(3))
df[F.NUMERIC].describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, col in zip(axes.ravel(), F.NUMERIC):
    sns.histplot(data=df, x=col, hue=F.TARGET, bins=20, ax=ax, palette={0: "#4c78a8", 1: "#e45756"})
plt.tight_layout()

In [ ]:
df[F.NUMERIC + [F.TARGET]].corr()[F.TARGET].drop(F.TARGET).sort_values().round(3)

Academic pressure (r ≈ 0.47) and financial stress (r ≈ 0.36) are the strongest numeric signals. Younger students
report depression more often, and CGPA is almost uncorrelated with the target.

In [ ]:
cats = ["gender", "sleep_duration", "dietary_habits", "family_history", "suicidal_thoughts"]
fig, axes = plt.subplots(1, 5, figsize=(17, 3.6), sharey=False)
for ax, c in zip(axes, cats):
    rate = df.groupby(c)[F.TARGET].mean().sort_values()
    rate.plot.barh(ax=ax, color="#4c78a8")
    ax.set_title(c); ax.set_xlim(0, 1); ax.set_ylabel(""); ax.set_xlabel("share with depression")
plt.tight_layout()

Poor sleep and an unhealthy diet go with a higher depression rate, gender makes no difference, and
**`suicidal_thoughts` is by far the strongest single feature** (79% vs 23%). This needs a decision, see below.

In [ ]:
vc = df["city"].value_counts()
rare_city = vc[vc < 100]
print(f"city: {len(vc)} distinct values, {len(rare_city)} with fewer than 100 rows ({rare_city.sum()} rows)")
print("examples:", list(rare_city.index[:12]))
vd = df["degree"].value_counts()
print(f"\ndegree: {len(vd)} distinct values, {(vd < 100).sum()} with fewer than 100 rows")

`city` contains junk entries (person names, `3.0`, `M.Tech`, ...) and `degree` has 28 levels. Rather than
hand-cleaning them, section 3 first checks whether `city`, `degree` and `gender` carry any signal at all.

### Feature-leakage check: `suicidal_thoughts`

The rules of the course ask to avoid target-derived features. Suicidal ideation is a separate survey answer and not
computed from the label, but it is closely tied to depression itself. We therefore train every model twice:
**`full`** (all features) and **`no_suicidal`** (without it) and report both. The deployed model uses `full`;
the README documents this trade-off.

## 3. Feature selection: do `city`, `degree` and `gender` help?

`gender` showed no relation with the target, `city` is full of junk and `degree` has many levels. We compare the
same logistic regression (same 5 folds) with and without these columns.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
import train as T

cv = StratifiedKFold(5, shuffle=True, random_state=T.SEED)
y = df[F.TARGET]
A = F.FEATURES_ALL_CANDIDATES
variants = {
    "all candidate features": A,
    "without city": [c for c in A if c != "city"],
    "without city + degree": [c for c in A if c not in ("city", "degree")],
    "without city + degree + gender (deployed)": F.FEATURES_FULL,
}
rows = {}
for name, cols in variants.items():
    pipe = Pipeline([("prep", T.make_preprocessor(cols)), ("clf", LogisticRegression(max_iter=2000, C=0.1))])
    s = cross_val_score(pipe, df[cols], y, cv=cv, scoring="roc_auc")
    rows[name] = {"n_features": len(cols), "cv_roc_auc": round(s.mean(), 4), "std_across_folds": round(s.std(), 4)}
pd.DataFrame(rows).T

Removing the columns changes the cross-validated AUC by at most 0.0001 (the fold-to-fold standard deviation
is about 0.003). The city-wise depression rates do differ (roughly 52% to 67%), but once the other answers are known
`city` adds nothing measurable. We therefore **drop `city`, `degree` and `gender`**: same quality, a simpler model and
a smaller API, and no need to handle the junk city values or unseen cities. The remaining
categorical columns (`sleep_duration`, `dietary_habits`, `family_history`, `suicidal_thoughts`) are one-hot encoded.

## 4. Models and hyper-parameter tuning

In [ ]:
metrics = json.loads(F.METRICS_PATH.read_text())   # produced by `python train.py`

def table(fs):
    keep = ["cv_roc_auc", "test_roc_auc", "test_accuracy", "test_precision", "test_recall", "test_f1"]
    rows = {m: {k: round(r[k], 4) for k in keep} | {"best_params": r["best_params"]}
            for m, r in metrics[fs]["models"].items()}
    return pd.DataFrame(rows).T

print("Feature set: full  ->  selected:", metrics["full"]["selected"])
table("full")

In [ ]:
print("Feature set: no_suicidal  ->  selected:", metrics["no_suicidal"]["selected"])
table("no_suicidal")

In [ ]:
pd.DataFrame({fs: {m: round(r["cv_roc_auc"], 4) for m, r in metrics[fs]["models"].items()}
              for fs in ("full", "no_suicidal")})

**Model selection.** Four models were tuned with a grid search (5-fold stratified CV on the 80% training split, scored by ROC AUC):
logistic regression (baseline), a shallow decision tree, a random forest and histogram gradient boosting.

* Logistic regression and gradient boosting are practically tied (CV AUC 0.9211 vs 0.9207); the random forest is
  slightly behind (0.916) and the single tree clearly (0.903). Some best parameters (tree depth, boosting learning
  rate and leaf size) sit at the edge of their grid; when we widened the tree / forest depth grids earlier, the CV AUC
  moved by less than 0.001, so we did not search further.
* We select the **logistic regression**: same accuracy, much simpler, faster, and its coefficients are interpretable.
* Dropping `suicidal_thoughts` costs about 0.05 AUC (≈ 0.92 → ≈ 0.87). Without it, logistic regression and gradient
  boosting are tied again (0.873 vs 0.874 CV AUC) and the forest and the tree trail them.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay
import train as T

assert metrics["full"]["selected"] == "logreg (baseline)"
cols = F.FEATURES_FULL
X, y = df[cols], df[F.TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=T.SEED)
best = metrics["full"]["models"]["logreg (baseline)"]["best_params"]
pipe = Pipeline([("prep", T.make_preprocessor(cols)),
                 ("clf", LogisticRegression(max_iter=2000, **best))]).fit(X_tr, y_tr)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_estimator(pipe, X_te, y_te, ax=axes[0])
ConfusionMatrixDisplay.from_estimator(pipe, X_te, y_te, ax=axes[1], cmap="Blues", colorbar=False)
plt.tight_layout()

In [ ]:
names = pipe.named_steps["prep"].get_feature_names_out()
coef = pd.Series(pipe.named_steps["clf"].coef_[0], index=names).sort_values()
print("Strongest negative (protective) and positive (risk) coefficients, standardised inputs:")
pd.concat([coef.head(7), coef.tail(7)]).round(3)

## 5. Conclusions and limitations

* A simple, interpretable logistic regression reaches **ROC AUC ≈ 0.92** and **recall ≈ 0.89** on held-out data.
* Largest risk coefficients: suicidal thoughts, academic pressure, financial stress, unhealthy diet, long
  study/work hours and short sleep. Protective: older age, healthy diet, study satisfaction. CGPA has almost no
  signal; `gender`, `city` and `degree` were tested and removed because they add nothing.
* The dataset is tagged synthetic and was collected in India, so the results say little about real populations.
  This model must not be used to diagnose anyone.